In [93]:
import dcm2niixpy
import os
import glob

class BIDSpet_name:
    def __init__(self, **kwargs):
        self.keys = ["sub", "ses", "task", "trc", "rec", "run", "desc"]
        self.values = kwargs
    
    def build(self):
        parts = []
        for key in self.keys:
            if key in self.values:
                parts.append(f"{key}-{self.values[key]}")
        parts.append("pet")
        return "_".join(parts)


# Example Usage
subject="PX142"
session="01"
bids_dir="/data_/mica3/MICA-PET/BIDS_mk6240"
subject_dir = os.path.join(f"{bids_dir}/sub-{subject}/ses-{session}")
pet_dir="/data_/mica3/MICA-PET/sourcedata_trc-mk6240_desc-REP/sub-PX142_ses-01_MK_1621_25Sep2024"
ecat_file = "*_EM_4D_MC01.v"
ecat = glob.glob(os.path.join(pet_dir, ecat_file))[0]
bids_name = BIDSpet_name(trc="mk6240", sub="01", ses="a", rec="acdyn").build()

print(ecat)
print(bids_name)
os.makedirs(subject_dir, exist_ok=True)

/data_/mica3/MICA-PET/sourcedata_trc-mk6240_desc-REP/sub-PX142_ses-01_MK_1621_25Sep2024/PX142-MK-HRRT4510-2024.9.25.16.21.5_EM_4D_MC01.v
sub-01_ses-a_trc-mk6240_rec-acdyn_pet


In [ ]:
# Run dcm2niix 
os.chdir(pet_dir)
dcm2niix = dcm2niixpy.DCM2NIIX(version="1.0.20241211")
dcm2niix.filename = bids_name
dcm2niix.bids_sidecar ="y"
dcm2niix.convert(ecat, output_path=subject_dir, options = ["-b y"])

singularity run --bind /data_/mica3/MICA-PET/sourcedata_trc-mk6240_desc-REP/sub-PX142_ses-01_MK_1621_25Sep2024/PX142-MK-HRRT4510-2024.9.25.16.21.5_EM_4D_MC01.v:/input --bind /data_/mica3/MICA-PET/BIDS_mk6240/sub-PX142/ses-01:/output docker://svdvoort/dcm2niix:1.0.20241211 -6 -a n -b y -ba y -d 5 -e n -f sub-01_ses-a_trc-mk6240_rec-acdyn_pet -g n -i n -l n -m 2 -r n -s n -t n -v 0 -w 2 -x n --big-endian o --progress n -z n -o /output /input


TypeError: join() argument must be str, bytes, or os.PathLike object, not 'NoneType'

In [163]:
import subprocess
import json

# Set the working directory to the script's location
__file__ = os.path.abspath('/host/yeatman/local_raid/rcruces/git_here/petpipe/petpipe')
repo_dir = os.path.dirname(os.path.abspath(__file__))

def merge_json_files(json_pet, json_subject):
    """
    Merge the newly created JSON file with an existing JSON file,
    overwriting the existing file with the merged content.

    Args:
        json_pet (str): Path to the newly created JSON file.
        json_subject (str): Path to the subject's JSON file.
    """
    # Load existing JSON if it exists
    if os.path.exists(json_subject):
        with open(json_subject, 'r') as f:
            existing_data = json.load(f)
    else:
        existing_data = {}

    # Load new JSON
    if os.path.exists(json_pet):
        with open(json_pet, 'r') as f:
            new_data = json.load(f)
    else:
        raise FileNotFoundError(f"New JSON file not found: {json_pet}")

    # Merge JSONs (existing data takes priority in case of conflicts)
    merged_data = {**new_data, **existing_data}

    # Overwrite the existing JSON file
    with open(json_pet, 'w') as f:
        json.dump(merged_data, f, indent=4)

    print(f"Updated JSON saved to: {json_pet}")

def convert_ecat_to_bids(in_file, out_file, output_dir, json=None):
    """
    Convert ECAT files to BIDS format using dcm2niix.
    Args:
        in_file (str): Input ECAT file path.
        out_file (str): Output BIDS file path.
        output_dir (str): Output directory for converted files.
    """
    
    command = f'dcm2niix -b y -v n -z y -o {output_dir} -f {out_file} {in_file}'
    print(command)
    try:
        result = subprocess.run(command, shell=True, check=True, capture_output=True, text=True)
        print(result.stdout)
        
        # Check if the output file was generated
        nifti_file = os.path.join(output_dir, f"{out_file}.nii.gz")
        if not os.path.isfile(nifti_file):
            print(f"Conversion failed, NIfTI file not generated: {nifti_file}")
            exit(1)
    except subprocess.CalledProcessError as e:
        print(f"Error running dcm2niix: {e.stderr}")
        exit(1)

    if json is not None:
        # Update JSON sidecar
        merge_json_files(os.path.join(output_dir, f"{out_file}.json"), json)


In [164]:
repo_dir

'/host/yeatman/local_raid/rcruces/git_here/petpipe'

In [165]:
pet_image = BIDSpet_name(trc="mk6240", sub=subject, ses=session, rec="acdyn").build()
convert_ecat_to_bids(f'{pet_dir}/*EM_4D_MC01.v', pet_image, subject_dir, json=os.path.join(repo_dir, "files/subject_trc-MK6240_pet.json"))

dcm2niix -b y -v n -z y -o /data_/mica3/MICA-PET/BIDS_mk6240/sub-PX142/ses-01 -f sub-PX142_ses-01_trc-mk6240_rec-acdyn_pet /data_/mica3/MICA-PET/sourcedata_trc-mk6240_desc-REP/sub-PX142_ses-01_MK_1621_25Sep2024/*EM_4D_MC01.v
Chris Rorden's dcm2niiX version v1.0.20241211  GCC13.3.0 x86-64 (64-bit Linux)
Saving ECAT as '/data_/mica3/MICA-PET/BIDS_mk6240/sub-PX142/ses-01/sub-PX142_ses-01_trc-mk6240_rec-acdyn_pet'
Conversion required 5.011926 seconds (4.587861 for core code).

Updated JSON saved to: /data_/mica3/MICA-PET/BIDS_mk6240/sub-PX142/ses-01/sub-PX142_ses-01_trc-mk6240_rec-acdyn_pet.json


In [166]:
# Create the mk6240 transmission
tx_image = BIDSpet_name(sub=subject, ses=session, desc="LinearAtenuationMap").build()
convert_ecat_to_bids(f"{pet_dir}/Transmission/*TX.v", tx_image, f"{subject_dir}")


dcm2niix -b y -v n -z y -o /data_/mica3/MICA-PET/BIDS_mk6240/sub-PX142/ses-01 -f sub-PX142_ses-01_desc-LinearAtenuationMap_pet /data_/mica3/MICA-PET/sourcedata_trc-mk6240_desc-REP/sub-PX142_ses-01_MK_1621_25Sep2024/Transmission/*TX.v
Chris Rorden's dcm2niiX version v1.0.20241211  GCC13.3.0 x86-64 (64-bit Linux)
Saving ECAT as '/data_/mica3/MICA-PET/BIDS_mk6240/sub-PX142/ses-01/sub-PX142_ses-01_desc-LinearAtenuationMap_pet'
Conversion required 0.268042 seconds (0.171494 for core code).



In [167]:
micapipe_dir="/data_/mica3/BIDS_MICs/derivatives/micapipe_v0.2.0"
t1_files = os.path.join(f"{micapipe_dir}/sub-{subject}/ses-01/anat/*_space-nativepro_T1w.*")

In [168]:
# Construct the path correctly
t1_files = glob.glob(os.path.join(micapipe_dir, f"sub-{subject}", "ses-01", "anat", "*_space-nativepro_T1w.*"))

print(t1_files)

['/data_/mica3/BIDS_MICs/derivatives/micapipe_v0.2.0/sub-PX142/ses-01/anat/sub-PX142_ses-01_space-nativepro_T1w.nii.gz', '/data_/mica3/BIDS_MICs/derivatives/micapipe_v0.2.0/sub-PX142/ses-01/anat/sub-PX142_ses-01_space-nativepro_T1w.json']


In [169]:
class BIDS_name:
    ALLOWED_SUFFIXES = ["FLAIR", "PDT2", "PDw", "T1w", "T2starw", "T2w", "UNIT1", "angio", "inplaneT1", "inplaneT2"]
    
    def __init__(self, **kwargs):
        self.keys = ["sub", "ses", "task", "acq", "ce", "rec", "run", "echo","part", "chunk", "suffix"]
        self.values = kwargs
        
        # Validate suffix
        if 'suffix' in self.values and self.values['suffix'] not in self.ALLOWED_SUFFIXES:
            raise ValueError(f"Invalid suffix: {self.values['suffix']}. Must be one of {self.ALLOWED_SUFFIXES}")
    
    def build(self):
        parts = []
        for key in self.keys:
            if key in self.values:
                if key == "suffix":
                    # Directly append the suffix without a prefix "suffix-"
                    suffix = self.values[key]
                    parts.append(suffix)
                elif isinstance(self.values[key], str):
                    parts.append(f"{key}-{self.values[key]}")
                elif isinstance(self.values[key], int):
                    parts.append(f"{key}-{self.values[key]}")
        
        return "_".join(parts)


In [170]:
t1_files = os.path.splitext(glob.glob(os.path.join(micapipe_dir, f"sub-{subject}", "ses-01", "anat", "*_space-nativepro_T1w.json"))[0])[0]
t1_str = BIDS_name(suffix="T1w", sub=subject, ses=session).build()

In [171]:
import shutil
shutil.copy2(f"{t1_files}.json", os.path.join(subject_dir, f"{t1_str}.json"))
shutil.copy2(f"{t1_files}.nii.gz", os.path.join(subject_dir, f"{t1_str}.nii.gz"))

'/data_/mica3/MICA-PET/BIDS_mk6240/sub-PX142/ses-01/sub-PX142_ses-01_T1w.nii.gz'

# Pet processing

In [172]:
print("\n-------------------------------------------------------------")
print("         PET pipeline - Processing")
print("-------------------------------------------------------------")
print(f"Subject: {subject}")
print(f"Session: {session}")
print(f"BIDS subject directory: {subject_dir}")


-------------------------------------------------------------
         PET pipeline - Processing
-------------------------------------------------------------
Subject: PX142
Session: 01
BIDS subject directory: /data_/mica3/MICA-PET/BIDS_mk6240/sub-PX142/ses-01
